In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.interpolate import RegularGridInterpolator
import seaborn as sns



def Price_path(S,r,sd,T,I,M):
    dt = 1 / M
    Z = np.random.normal(0, np.sqrt(dt), size=(I, int(T * M)))
    x = np.zeros((I, int(T * M) + 1))
    x[:, 0] = S
    x[:, 1:] = S * np.exp(np.cumsum((r - 0.5 * sd ** 2) * dt + sd * Z, axis=1))
    return x

def Lag_3(X):
    L_0 = np.exp(-X / 2)
    L_1 = np.exp(-X / 2) * (1 - X)
    L_2 = np.exp(-X / 2) * (1 - 2 * X + X ** 2 / 2)
    return np.column_stack((L_0, L_1, L_2))

def CRR(S0, K, T, r, sd, N):
    dt = T / N
    
    u = np.exp(sd * np.sqrt(dt))
    d = np.exp(-sd * np.sqrt(dt))
    if u == d:
        return 0.0, 0.0, 0.0
    q = (np.exp(r * dt) - d) / (u - d)
    df = np.exp(-r * dt)

    
    S = S0 * d ** (np.arange(N, -1, -1)) * u ** (np.arange(0, N + 1, 1))
    C = np.maximum(K - S, 0) 

    
    S_tree = {}
    C_tree = {}

    for i in np.arange(N - 1, -1, -1):
        S = S0 * d ** (np.arange(i, -1, -1)) * u ** (np.arange(0, i + 1, 1))
        continuation = df * (q * C[1:i + 2] + (1 - q) * C[0:i + 1])
        exercise = np.maximum(K - S, 0)
        C = np.maximum(continuation, exercise)

        
        if i == 1:
            C_tree[1] = C.copy()
            S_tree[1] = S.copy()
        elif i == 2:
            C_tree[2] = C.copy()
            S_tree[2] = S.copy()

    
    C_up = C_tree[1][0]
    C_down = C_tree[1][1]
    S_up = S_tree[1][0]
    S_down = S_tree[1][1]
    delta = (C_up - C_down) / (S_up - S_down)

    C_uu = C_tree[2][0]
    C_ud = C_tree[2][1]
    C_dd = C_tree[2][2]
    S_uu = S_tree[2][0]
    S_ud = S_tree[2][1]
    S_dd = S_tree[2][2]
    gamma = ((C_uu - C_ud) / (S_uu - S_ud) - (C_ud - C_dd) / (S_ud - S_dd)) / ((S_uu - S_dd) / 2)

    return C[0], delta, gamma


def Delta_Interpol(delta, s0, time_grid, T, N, K, I, S):
    interp_f = RegularGridInterpolator((s0, time_grid), delta, method='linear', bounds_error=True, fill_value=None)
    neg_one_vec = np.zeros(I) - 1
    Delta_M = np.zeros_like(S)

    for i in range(0, T * N):
        t = time_grid[i] 
        

        in_range = (S[:, i] >= K - 10) & (S[:, i] <= K + 20) 
        notin_range = (S[:, i] < K - 10) 

        
        if S[notin_range, i].size > 0:
            Delta_M[notin_range, i] = neg_one_vec[notin_range]

        S_in_range = S[in_range, i]
        points = np.array([S_in_range, np.full_like(S_in_range, t)]).T
        Delta_M[in_range, i] = interp_f(points)
    return Delta_M



def LSM_put_bound(S,K,r,sd,T,I,M):
    dt = 1 / M
    price = Price_path(S,r,sd,T,I,M) / K
    h = np.maximum(1 - price, 0)
    h[:, 0] = 0
    h_copy = h.copy()

    disc = np.exp(-r * dt)

    for i in range(T * M, 0, -1):
        ITM = h[:, i - 1] > 0
        if not ITM.any():  
            continue

        Y = disc * h_copy[ITM, i]
        X = price[ITM, i - 1]
        Z = Lag_3(X)

        reg = LinearRegression()
        reg.fit(Z, Y)
        a, b, c, d = reg.intercept_, *reg.coef_

        Q = price[:, i - 1]
        E = a + b * Lag_3(Q)[:, 0] + c * Lag_3(Q)[:, 1] + d * Lag_3(Q)[:, 2]

        h[:, i - 1] = np.where(h[:, i - 1] < E, 0, h[:, i - 1])
        h[h[:, i - 1] > 0, i:] = 0
        h_copy[:, i - 1] = np.where(h[:, i - 1] == 0, h_copy[:, i] * disc, h_copy[:, i - 1])

    cont_bound = []

    for i in range(T*M):
        has_cashflow = h[:,i] > 0
        if np.any(h[:,i] > 0) == True:
            cont_bound_element = (K - np.min(h[:,i][has_cashflow]) * K)
        else:
            cont_bound_element = np.nan
        cont_bound.append(cont_bound_element)

    cont_bound.append(40)
    return cont_bound



In [ ]:
S0 = 40
I0 = 10**5
K0 = 40
N0 = 50
dt = 1/N0
sd0 = 0.2
T0 = 1
r0 = 0.06
interest_rate = np.exp(r0*dt)
n0 = 20


price = CRR(S0,K0,T0,r0,sd0,10**3)[0]


Cont_bound = LSM_put_bound(S0,K0,r0,sd0,T0,I0,N0)


s0 = np.linspace(30, 60, n0 + 1)

time_grid = np.linspace(0, T0 * N0 - 1, T0 * N0, dtype=int)

delta_m = np.array([[CRR(s, K0, T0-t/N0,r0,sd0,10**3)[1] for s in s0] for t in time_grid]).T


Stock_paths = Price_path(S0, r0,sd0,T0, I0,N0)


Delta = Delta_Interpol(delta_m, s0, time_grid, T0, N0, K0, I0, Stock_paths)


boundary = (Stock_paths <= Cont_bound)
boundary &= np.cumsum(boundary, axis=1) == 1
before_boundary = np.cumsum(boundary, axis=1) == 0


option_CFM = np.zeros_like(Stock_paths)
option_CFM[boundary] = (K0 - Stock_paths)[boundary]


replicating_pf = np.zeros_like(Stock_paths)
replicating_pf[:,0] = price



riskneutral_position = np.zeros_like(Stock_paths)
riskneutral_position[:,0] = replicating_pf[:,0] - Delta[:,0] * Stock_paths[:,0]

for i in range(T0*N0):
    replicating_pf[:,i+1] = riskneutral_position[:,i] * interest_rate + Delta[:,i] * Stock_paths[:,i+1]
    riskneutral_position[:, i+1] = replicating_pf[:, i+1] - Delta[:,i+1] * Stock_paths[:,i+1]



hedge_errors = np.zeros_like(Stock_paths)
hedge_errors[boundary] = (replicating_pf - option_CFM)[boundary]
notexercised = np.sum(boundary, axis=1) == 0
hedge_errors[notexercised,N0] = (replicating_pf - (option_CFM))[notexercised,N0]

disc_vector = np.vander([interest_rate],N0+1).T

disc_hedge_errors = (hedge_errors @ disc_vector)[:,0]

#Hedge error
plt.figure() 
plt.grid(alpha=0.5, linestyle = '--', color = 'black', zorder = 1)
ax = plt.gca()
plt.hist(disc_hedge_errors, bins=50, density=True, edgecolor='black')
sns.kdeplot(disc_hedge_errors, linewidth=1)
plt.xlim(-2.5, 2.5)
plt.ylabel("")
plt.tight_layout()

print(
np.mean(hedge_errors),
np.var(hedge_errors),
np.min(hedge_errors),
np.max(hedge_errors)
)

print(
np.mean(disc_hedge_errors),
np.var(disc_hedge_errors),
np.min(disc_hedge_errors),
np.max(disc_hedge_errors)
)



In [ ]:
def Gamma_Interpol(gamma_grid, s0, time_grid, T, N, K, I, S):
    interp_g = RegularGridInterpolator((s0, time_grid), gamma_grid, method='linear', bounds_error=True, fill_value=None)
    gamma_matrix = np.zeros_like(S)
    
    for i in range(0, T * N):
        t = time_grid[i]
        in_range = (S[:, i] >= K - 10) & (S[:, i] <= K + 20)
        S_in_range = S[in_range, i]
        points = np.array([S_in_range, np.full_like(S_in_range, t)]).T
        gamma_matrix[in_range, i] = interp_g(points)
    return gamma_matrix

gamma_m = np.array([[CRR(s, K0, T0-t/N0, r0, sd0, 10**3)[2] for s in s0] for t in time_grid]).T

Gamma = Gamma_Interpol(gamma_m, s0, time_grid, T0, N0, K0, I0, Stock_paths)

avg_gamma_path = np.mean(Gamma, axis=1)

plt.figure(figsize=(9, 6))
plt.grid(alpha=0.5, linestyle='--', color='black')
plt.scatter(avg_gamma_path, disc_hedge_errors, alpha=0.4, color="blue", edgecolors='k', linewidth=0.2)
plt.xlabel("Average Gamma Along Path", fontsize=14)
plt.ylabel("Hedge Error", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
def Poly_8(X):
    P_0 = X**0
    P_1 = X
    P_2 = X**2
    P_3 = X**3
    P_4 = X**4
    P_5 = X**5
    P_6 = X**6
    P_7 = X**7
    P_8 = X**8
    return np.column_stack((P_0,P_1,P_2,P_3,P_4,P_5,P_6,P_7,P_8))

def Ker(x):
    return 2 * np.sin(np.arcsin(2 * x - 1) / 3)

def LSM_ISD(S0, K, r, sd, T, I, M, alpha, alpha_opt, n):
    
    prices = []
    deltas = []
    gammas = []
    for i in range(n):
        dt = T/M
        # M=int(M*T)
        M = max(2, int(M * T))
        df = np.exp(-r*dt)
        S = np.zeros((I, M + 1))
        U = np.random.uniform(0, 1, I)
        S_init = S0 + alpha*Ker(U)  
        S[:, 0] = S_init
        S[:, 1:] = (S_init)[:, np.newaxis] * np.exp(np.cumsum((r - 0.5 * sd ** 2) * dt + sd * np.random.normal(0, np.sqrt(dt), size=(I, M)), axis=1))

        deviations = S[:, 0] - S0
        mask = np.abs(deviations) <= alpha_opt
        count = np.sum(mask)

        if count > 0:
            U_new = np.random.uniform(0, 1, count)
            new_disp = alpha_opt * Ker(U_new)
            S_new0 = S0 + new_disp
            
            valid_mask = np.abs(S_new0 - S0) <= alpha_opt
            valid_count = np.sum(valid_mask)
        
            replace_indices = np.where(mask)[0][:valid_count]
            
            S[replace_indices, 0] = S_new0[valid_mask]
            S[replace_indices, 1:] = S_new0[valid_mask][:, np.newaxis] * np.exp(np.cumsum((r - 0.5 * sd ** 2) * dt + sd * np.random.normal(0, np.sqrt(dt), size=(valid_count, M)), axis=1))


        s = S/K 
        h = np.maximum(1 - s, 0)
        h[:, 0] = 0 
        h_copy = h.copy()

        for i in range(M, 1, -1):
            ITM = h[:, i - 1] > 0
            if np.sum(ITM) == 0:
                continue  

            Y = df * h_copy[ITM, i]
            X = s[ITM, i - 1]
            Z = Poly_8(X)

            reg1 = LinearRegression().fit(Z, Y)

            basis = Poly_8(s[:, i - 1])
            cont = reg1.predict(basis)

            h[:, i - 1] = np.where(h[:, i - 1] <= cont, 0, h[:, i - 1])
            h[h[:, i - 1] > 0, i:] = 0
            h_copy[:, i - 1] = np.where(h[:, i - 1] == 0, h_copy[:, i] * df, h_copy[:, i - 1])

        df_vec = np.power(df, np.arange(M + 1)-1).reshape(-1, 1)

        disc_CashFlows = h @ df_vec

        Z_1 = disc_CashFlows.flatten()
        
        X_poly = Poly_8(s[:,1]) 
        
        
        reg2 = LinearRegression().fit(X_poly, Z_1)

        value_t1 = reg2.predict(X_poly)

        value = np.maximum(value_t1, np.maximum(1-s[:,1],0))* df * K



        
        outside_opt_mask = (S[:, 0] - S0 < -alpha_opt) | (S[:, 0] - S0 > alpha_opt)

        
        n_resim = np.sum(outside_opt_mask)


        if n_resim > 0:
            U_new = np.random.uniform(0, 1, n_resim)
            S_init_new = S0 + alpha_opt * Ker(U_new)
            
            S_new = np.zeros((n_resim, M + 1))
            S_new[:, 0] = S_init_new
            S_new[:, 1:] = (S_init_new)[:, np.newaxis] * np.exp(np.cumsum(
                (r - 0.5 * sd ** 2) * dt + sd * np.random.normal(0, np.sqrt(dt), size=(n_resim, M)), axis=1))
            
            s_new = S_new / K
            h_new = np.maximum(1 - s_new, 0)
            h_new[:, 0] = 0
            h_copy_new = h_new.copy()

            for j in range(M, 1, -1):
                ITM = h_new[:, j - 1] > 0
                if np.sum(ITM) == 0:
                    continue
                Y = df * h_copy_new[ITM, j]
                X = s_new[ITM, j - 1]
                Z = Poly_8(X)
                reg1 = LinearRegression().fit(Z, Y)
                basis = Poly_8(s_new[:, j - 1])
                cont = reg1.predict(basis)
                h_new[:, j - 1] = np.where(h_new[:, j - 1] <= cont, 0, h_new[:, j - 1])
                h_new[h_new[:, j - 1] > 0, j:] = 0
                h_copy_new[:, j - 1] = np.where(h_new[:, j - 1] == 0, h_copy_new[:, j] * df, h_copy_new[:, j - 1])

            disc_CF_new = h_new @ df_vec
            Z_1_new = disc_CF_new.flatten()
            X_poly_new = Poly_8(s_new[:, 1])
            reg2_new = LinearRegression().fit(X_poly_new, Z_1_new)
            value_t1_new = reg2_new.predict(X_poly_new)
            value_new = np.maximum(value_t1_new, np.maximum(1 - s_new[:, 1], 0)) * df * K

            value[outside_opt_mask] = value_new
            S[outside_opt_mask, 0] = S_new[:, 0]

        X_poly2 = Poly_8(S[:, 0] - S0)
        reg3 = LinearRegression().fit(X_poly2, value)


        price = reg3.intercept_
        prices.append(price)

        delta = reg3.coef_[1]
        deltas.append(delta)

        gamma = 2* reg3.coef_[2]
        gammas.append(gamma)
    avg_price = np.mean(prices)    
    avg_delta = np.mean(deltas)
    avg_gamma = np.mean(gammas)




    return avg_delta, avg_gamma

In [ ]:
S0 = 40
I0 = 10**5
K0 = 40
N0 = 50
dt = 1/N0
sd0 = 0.2
T0 = 1
r0 = 0.06
interest_rate = np.exp(r0*dt)
n0 = 20


price = CRR(S0,K0,T0,r0,sd0,10**3)[0]


Cont_bound = LSM_put_bound(S0,K0,r0,sd0,T0,I0,N0)


s0 = np.linspace(30, 60, n0 + 1)

time_grid = np.linspace(0, T0 * N0 - 1, T0 * N0, dtype=int)

delta_m_2stage = np.array([[LSM_ISD(s, K0,r0,sd0,T0-t/N0, I0,N0,25,5,1)[0] for s in s0] for t in time_grid]).T


Stock_paths = Price_path(S0, r0,sd0,T0, I0,N0)


Delta = Delta_Interpol(delta_m_2stage, s0, time_grid, T0, N0, K0, I0, Stock_paths)


boundary = (Stock_paths <= Cont_bound)
boundary &= np.cumsum(boundary, axis=1) == 1
before_boundary = np.cumsum(boundary, axis=1) == 0


option_CFM = np.zeros_like(Stock_paths)
option_CFM[boundary] = (K0 - Stock_paths)[boundary]


replicating_pf = np.zeros_like(Stock_paths)
replicating_pf[:,0] = price



riskneutral_position = np.zeros_like(Stock_paths)
riskneutral_position[:,0] = replicating_pf[:,0] - Delta[:,0] * Stock_paths[:,0]

for i in range(T0*N0):
    replicating_pf[:,i+1] = riskneutral_position[:,i] * interest_rate + Delta[:,i] * Stock_paths[:,i+1]
    riskneutral_position[:, i+1] = replicating_pf[:, i+1] - Delta[:,i+1] * Stock_paths[:,i+1]



hedge_errors = np.zeros_like(Stock_paths)
hedge_errors[boundary] = (replicating_pf - option_CFM)[boundary]
notexercised = np.sum(boundary, axis=1) == 0
hedge_errors[notexercised,N0] = (replicating_pf - (option_CFM))[notexercised,N0]

disc_vector = np.vander([interest_rate],N0+1).T

disc_hedge_errors = (hedge_errors @ disc_vector)[:,0]

#Hedge error
plt.figure() 
plt.grid(alpha=0.5, linestyle = '--', color = 'black', zorder = 1)
ax = plt.gca()
# ax.set_axisbelow(True)
plt.hist(disc_hedge_errors, bins=50, density=True, edgecolor='black')
sns.kdeplot(disc_hedge_errors, linewidth=1)
plt.xlim(-4, 4)
plt.ylabel("")
plt.tight_layout()

print(
np.mean(hedge_errors),
np.var(hedge_errors),
np.min(hedge_errors),
np.max(hedge_errors)
)

print(
np.mean(disc_hedge_errors),
np.var(disc_hedge_errors),
np.min(disc_hedge_errors),
np.max(disc_hedge_errors)
)



In [ ]:
gamma_m_2stage = np.array([[LSM_ISD(s, K0,r0,sd0,T0-t/N0, I0,N0,25,5,1)[1] for s in s0] for t in time_grid]).T

Gamma = Gamma_Interpol(gamma_m_2stage, s0, time_grid, T0, N0, K0, I0, Stock_paths)

avg_gamma_path = np.mean(Gamma, axis=1)

plt.figure(figsize=(9, 6))
plt.grid(alpha=0.5, linestyle='--', color='black')
plt.scatter(avg_gamma_path, disc_hedge_errors, alpha=0.4, color="blue", edgecolors='k', linewidth=0.2)
plt.xlabel("Average Gamma Along Path", fontsize=14)
plt.ylabel("Hedge Error", fontsize=14)
plt.tight_layout()
plt.show()